# MLflow — Three Ways to Track an Experiment

This notebook demonstrates the same logging task using three different MLflow APIs:

1. **Fluent API** with a context manager (`with mlflow.start_run():`) — recommended for scripts.
2. **Fluent API** imperatively (`mlflow.start_run()` / `mlflow.end_run()`) — convenient across notebook cells.
3. **Client API** (`MlflowClient`) — explicit, stateless, used for automation and management.

All three produce equivalent runs in the MLflow UI.

## Setup

Shared imports and the path to the artifact we'll log.

In [2]:
import mlflow
from mlflow.tracking import MlflowClient

# model_path = r"E:\SDA\Courses\AIRemote\done_AIRemoteRO9\done_before\deep_learning (16)\deep_learning\model.h5"

## 1. Fluent API — context manager

The `with` block opens a run, makes it the active run for the thread, and **auto-closes it** on exit (sets status to `FAILED` if an exception is raised). This is the recommended style for training scripts.

In [3]:
mlflow.set_experiment("Experiment - Fluent API - context manager")

with mlflow.start_run() as run:
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_metric("accuracy", 0.95)
     #mlflow.log_artifact(model_path)
    print("Run ID:", run.info.run_id)
# Run is automatically ended here.

2026/05/14 19:42:14 INFO mlflow.tracking.fluent: Experiment with name 'Experiment - Fluent API - context manager' does not exist. Creating a new experiment.


Run ID: cef7e55806fd41ca94e0c2e07c2bc79a


## 2. Fluent API — imperative (direct function calls)

Same Fluent API, but without the context manager. Useful in notebooks when you want to log across multiple cells. **You must call `mlflow.end_run()` yourself** — otherwise the run stays open and the next `start_run()` will raise.

In [4]:
mlflow.set_experiment("Experiment - Fluent API - direct function call")

run = mlflow.start_run()
print(run.info)

2026/05/14 19:42:19 INFO mlflow.tracking.fluent: Experiment with name 'Experiment - Fluent API - direct function call' does not exist. Creating a new experiment.


<RunInfo: artifact_uri='file:///e:/airemote9-course/sda-airemote-9/course_3/mlruns/477458141222527396/2ba00aa067824bdfb7b4f9d3a4ba479e/artifacts', end_time=None, experiment_id='477458141222527396', lifecycle_stage='active', run_id='2ba00aa067824bdfb7b4f9d3a4ba479e', run_name='classy-fawn-347', run_uuid='2ba00aa067824bdfb7b4f9d3a4ba479e', start_time=1778776939136, status='RUNNING', user_id='Fabian'>


In [5]:
mlflow.log_param("learning_rate", 0.01)
mlflow.log_metric("accuracy", 0.95)
#mlflow.log_artifact(model_path)

In [37]:
mlflow.end_run()  # must be called manually

## 3. Client API — explicit, stateless

`MlflowClient` has **no concept of an active run** — every call takes an explicit `run_id`. Note the method names differ from the Fluent API:

| Fluent | Client |
|---|---|
| `mlflow.start_run()` | `client.create_run(experiment_id)` |
| `mlflow.end_run()` | `client.set_terminated(run_id)` |

We also use `get_experiment_by_name` first so re-running the cell doesn't fail when the experiment already exists.

In [6]:
client = MlflowClient()

# Get-or-create the experiment (create_experiment alone fails on re-run)
exp_name = "Experiment - Client API"
exp = client.get_experiment_by_name(exp_name)
experiment_id = exp.experiment_id if exp else client.create_experiment(exp_name)

# Create a run (NOT start_run — that doesn't exist on MlflowClient)
run = client.create_run(experiment_id=experiment_id)
run_id = run.info.run_id
print("Run ID:", run_id)

Run ID: b3b7f7a21e6742e195ca6946fdc72bf3


In [7]:
client.log_param(run_id, "learning_rate", 0.01)
client.log_metric(run_id, "accuracy", 0.95)
#client.log_artifact(run_id, model_path)
# Terminate the run (NOT end_run — that doesn't exist on MlflowClient)
client.set_terminated(run_id)

## Summary

| Section | API | Run lifecycle | Best for |
|---|---|---|---|
| 1 | Fluent (context manager) | Auto-closes on `with` exit | Training scripts |
| 2 | Fluent (imperative) | Manual `end_run()` | Multi-cell notebook flows |
| 3 | Client | Manual `set_terminated(run_id)` | Automation, registry ops, search |

Open the MLflow UI with `mlflow ui` from the working directory to inspect all three experiments.

## Query runs


In [8]:
# Set the experiment (replace with your experiment name or ID)
experiment_name = "Experiment - Client API"
experiment = mlflow.get_experiment_by_name(experiment_name)

# Search for runs with a specific metric and parameter
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="metrics.accuracy > 0.90 and params.learning_rate = '0.01'",
    order_by=["metrics.accuracy DESC"],
    max_results=5
)

# Display the results
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.accuracy,params.learning_rate,tags.mlflow.runName
0,b3b7f7a21e6742e195ca6946fdc72bf3,118609479659508771,FINISHED,file:///e:/airemote9-course/sda-airemote-9/cou...,2026-05-14 16:42:23.697000+00:00,2026-05-14 16:42:26.128000+00:00,0.95,0.01,bedecked-midge-678
